# Baseline Model — Fraud Detection

## Objective

This notebook trains the project's first model on the engineered features
from `04_feature_engineering.ipynb`. The goal here is deliberately narrow,
per `17_CLASSIFICATION`: establish a simple, interpretable **baseline** —
logistic regression — not to find the best possible model. Everything more
sophisticated (gradient-boosted trees, resampling strategies, extensive
tuning) belongs in `06_model_experiments.ipynb`, where it can be measured
*against* this baseline rather than in a vacuum.

Three things this notebook is strict about, each tied back to an earlier
knowledge-layer section:

1. **Time-based split, not random** (`16_DATA_LEAKAGE_AND_TIME`) — we
   carve a validation set out of the *end* of the training timeline, never
   a random sample, so validation performance actually reflects how the
   model would perform on transactions it hasn't seen yet.
2. **Class imbalance is handled explicitly, not ignored** (`18_CLASS_IMBALANCE`)
   — via `class_weight="balanced"`, the simplest of the three approaches
   discussed in the knowledge layer. Resampling (SMOTE) is left for
   `06_model_experiments.ipynb` to compare against this.
3. **Evaluation uses precision/recall/PR-AUC, not accuracy**
   (`19_MODEL_EVALUATION`) — accuracy would be close to meaningless given
   how rare fraud is in this dataset.


## Setup


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_recall_curve, RocCurveDisplay,
)
import joblib


In [ ]:
train_fe = pd.read_pickle(r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\data\02_intermediate\train_features.pkl')


In [ ]:
test_fe = pd.read_pickle(r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\data\02_intermediate\test_features.pkl')


In [ ]:
print("train_fe shape:", train_fe.shape)
print("test_fe shape: ", test_fe.shape)
print("\nFraud rate in train_fe:", train_fe["isFraud"].mean().round(5))


## Step 1: Select the Feature Set

We deliberately exclude a few categories of column from the baseline:

- **Identifiers** (`TransactionID`) — carry no generalizable signal and
  would just let the model memorize individual rows.
- **Raw categorical strings** (`ProductCD`, `card4`, `card6`, `DeviceType`,
  `P_emaildomain`, `R_emaildomain`) — these were already converted into
  `*_freq_enc` numeric columns in `04_feature_engineering.ipynb`. Feeding
  the raw strings into logistic regression would require one-hot encoding
  we haven't built yet, so for this baseline we use the frequency-encoded
  versions only.
- **Raw identity columns** (`id_01`, `id_02`, ...) — mostly missing (see
  `02_data_validation.ipynb`), and already summarized by the
  `has_identity` flag from feature engineering. Using the raw columns here
  would require an imputation strategy we haven't decided on yet; that's
  a reasonable thing to revisit in `06_model_experiments.ipynb`.
- **The target itself** (`isFraud`), obviously.

Everything else — numeric raw fields plus every engineered feature from
notebook 04 — goes into the baseline.


In [ ]:
exclude_cols = {"TransactionID", "isFraud"}
raw_categorical_cols = {"ProductCD", "card4", "card6", "DeviceType",
                         "P_emaildomain", "R_emaildomain"}
raw_identity_cols = {c for c in train_fe.columns if c.startswith("id_")}

exclude_cols |= raw_categorical_cols
exclude_cols |= raw_identity_cols

feature_cols = [c for c in train_fe.columns if c not in exclude_cols]

# keep only numeric columns -- a final defensive filter in case anything
# non-numeric slipped through
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train_fe[c])]

print(f"Using {len(feature_cols)} features:")
print(feature_cols)


## Step 2: Time-Based Train / Validation Split

`test_fe` has no `isFraud` label — it's the true holdout set this project
would eventually be scored against, the same way a Kaggle-style
competition test set works. To actually measure how well a model
performs, we need labelled data it hasn't seen, which means carving a
validation set out of `train_fe` itself.

**Critically, this split is by time, not at random.** We sort by
`TransactionDT` and take the most recent 20% as validation. A random
split would let the model validate on transactions that occurred
*earlier* than some of its training data — exactly the leakage problem
covered in `16_DATA_LEAKAGE_AND_TIME` — and would make this baseline look
better than it will actually perform on genuinely future transactions.


In [ ]:
train_sorted = train_fe.sort_values("TransactionDT").reset_index(drop=True)

split_idx = int(len(train_sorted) * 0.8)
split_time = train_sorted.iloc[split_idx]["TransactionDT"]

train_split = train_sorted.iloc[:split_idx]
val_split = train_sorted.iloc[split_idx:]

print(f"Split point: TransactionDT = {split_time}")
print(f"Train rows: {len(train_split)}  ({train_split['TransactionDT'].min()} - {train_split['TransactionDT'].max()})")
print(f"Val rows:   {len(val_split)}  ({val_split['TransactionDT'].min()} - {val_split['TransactionDT'].max()})")
print(f"\nFraud rate -- train split: {train_split['isFraud'].mean():.4%}")
print(f"Fraud rate -- val split:   {val_split['isFraud'].mean():.4%}")


**Worth checking before moving on:** if the two fraud rates above differ
wildly, that's consistent with the fraud-rate-drift pattern explored in
`03_eda.ipynb` section 4 (temporal behaviour) — not necessarily a bug, but
worth keeping in mind when interpreting validation performance.


In [ ]:
X_train = train_split[feature_cols]
y_train = train_split["isFraud"]

X_val = val_split[feature_cols]
y_val = val_split["isFraud"]


## Step 3: Scale Features

Logistic regression is sensitive to feature scale — a feature ranging in
the thousands (like `card1_amt_sum_24h`) would dominate one ranging in
single digits (like `day_of_week`) purely due to units, not actual
importance. We standardize every feature to zero mean and unit variance.

**The scaler is fit on the training split only**, then applied to
validation — fitting it on validation (or on the combined data) would let
validation-set statistics leak into how training features are scaled,
a subtler cousin of the leakage problem from Step 2.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


## Step 4: Train the Baseline Model

A plain logistic regression, with `class_weight="balanced"` so the loss
function penalizes missing a fraud case more heavily than a false alarm,
roughly in proportion to how rare fraud is — the simplest of the three
class-imbalance approaches from `18_CLASS_IMBALANCE`.


In [ ]:
baseline_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
)

baseline_model.fit(X_train_scaled, y_train)
print("Model trained.")


## Step 5: Evaluate on the Validation Split

Per `19_MODEL_EVALUATION`, we lead with precision, recall, and PR-AUC
rather than accuracy. We still compute ROC-AUC for reference, but we
don't treat it as the headline number.


In [ ]:
val_probs = baseline_model.predict_proba(X_val_scaled)[:, 1]
val_preds_default = (val_probs >= 0.5).astype(int)

print("Classification report at default 0.5 threshold:")
print(classification_report(y_val, val_preds_default, digits=4))

**Why the 0.5 threshold is only a starting reference, not a real
decision:** nothing about 0.5 is special for this problem — it's simply
scikit-learn's default. The actual threshold a production system would
use depends on the cost tradeoff from `12_BUSINESS_PROBLEM`, and gets
chosen deliberately in `08_risk_scoring.ipynb`, not assumed here.


In [ ]:
roc_auc = roc_auc_score(y_val, val_probs)
pr_auc = average_precision_score(y_val, val_probs)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC (average precision): {pr_auc:.4f}")
print(
    "\nPR-AUC is the more trustworthy of the two here -- with this much "
    "class imbalance, ROC-AUC can look deceptively strong even when "
    "precision at a usable recall level is quite poor (see 19_MODEL_EVALUATION)."
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Confusion matrix
cm = confusion_matrix(y_val, val_preds_default)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted: Legit", "Predicted: Fraud"],
            yticklabels=["Actual: Legit", "Actual: Fraud"], ax=axes[0])
axes[0].set_title("Confusion Matrix (threshold = 0.5)")

# Precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
axes[1].plot(recalls, precisions, color="firebrick")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title(f"Precision-Recall Curve (PR-AUC = {pr_auc:.3f})")

plt.tight_layout()
plt.show()

## Step 6: Threshold Exploration

A quick look at how precision and recall trade off at a few candidate
thresholds, as groundwork for `08_risk_scoring.ipynb` and
`21_DECISION_ENGINE` later. This is exploratory only — we are not
selecting a production threshold in this notebook.


In [ ]:
threshold_table = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    preds_t = (val_probs >= t).astype(int)
    tp = ((preds_t == 1) & (y_val == 1)).sum()
    fp = ((preds_t == 1) & (y_val == 0)).sum()
    fn = ((preds_t == 0) & (y_val == 1)).sum()
    precision_t = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall_t = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    threshold_table.append({
        "threshold": t, "flagged": int((preds_t == 1).sum()),
        "precision": round(precision_t, 4), "recall": round(recall_t, 4),
    })

pd.DataFrame(threshold_table)


**How to read this table:** moving the threshold down catches more fraud
(higher recall) at the cost of more false alarms (lower precision), and
vice versa. Which tradeoff is "better" is a business question from
`12_BUSINESS_PROBLEM`, not something this notebook can answer on its own.


## Step 7: Feature Importance (Coefficients)

Because every feature was standardized in Step 3, logistic regression
coefficients are directly comparable to each other — a larger absolute
coefficient means that feature has a stronger linear relationship with
the predicted log-odds of fraud, holding the others constant.

This is a useful sanity check against the correlation ranking from
`04_feature_engineering.ipynb` Step 9, but it is not full explainability
in the sense of `22_EXPLAINABILITY` — a coefficient describes the model's
*global* behaviour, not why any one specific transaction was flagged.
That per-transaction explanation is what `09_explainability.ipynb` is for.


In [ ]:
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": baseline_model.coef_[0],
}).sort_values("coefficient", key=lambda s: s.abs(), ascending=False)

print(coef_df.to_string(index=False))


In [ ]:
plt.figure(figsize=(8, max(4, len(coef_df) * 0.3)))
top_coefs = coef_df.head(20)
colors = ["firebrick" if c > 0 else "steelblue" for c in top_coefs["coefficient"]]
plt.barh(top_coefs["feature"], top_coefs["coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Standardized coefficient")
plt.title("Top 20 Features by |Coefficient| (red = pushes toward fraud)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Step 8: Save the Baseline Model

We save the trained model, the fitted scaler, and the exact feature list
together — all three are required to reproduce a prediction consistently,
and saving them separately risks them drifting out of sync later.


In [ ]:
model_artifact = {
    "model": baseline_model,
    "scaler": scaler,
    "feature_cols": feature_cols,
    "trained_at_split_time": split_time,
}

model_out_path = r'C:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model\models\baseline_logistic_regression.joblib'
joblib.dump(model_artifact, model_out_path)

print("Saved baseline model artifact to:")
print(" ", model_out_path)


## Summary

Trained and evaluated the project's first model:

- **Model:** logistic regression, `class_weight="balanced"`, on the
  frequency-encoded and behaviourally-engineered features from
  `04_feature_engineering.ipynb`.
- **Split:** time-based (most recent 20% of train held out as
  validation), not random — protecting against the leakage pattern in
  `16_DATA_LEAKAGE_AND_TIME`.
- **Evaluation:** led with PR-AUC and the precision/recall tradeoff
  across thresholds, rather than accuracy, per `19_MODEL_EVALUATION`.
- **Saved:** model, scaler, and feature list bundled together in
  `models/baseline_logistic_regression.joblib`.

**Read your own numbers once this runs on the real data**, and treat them
as the number to beat, not a final result: the whole point of a baseline
is to give `06_model_experiments.ipynb` something concrete to improve on
— with stronger models (gradient-boosted trees), other imbalance
strategies (SMOTE), and a one-hot or target-encoded treatment of the raw
categorical columns we deliberately left out here.

**Next step:** `06_model_experiments.ipynb`.
